# Orthodontic Treatment Velocity Synthetic Dataset

This notebook creates a synthetic patient dataset for cohort analysis in an orthodontic treatment dashboard.

# Define the Dataset Columns

The synthetic dataset should include the fields needed to support cohort analysis, treatment drift, retention trends, demographic comparison, and delay drilldown in Tableau.

## Core patient information
- `Patient_ID`
- `Clinic_Location`
- `Start_Date`
- `Age_Group`

## Treatment timeline fields
- `Projected_Treatment_Weeks`
- `Projected_Completion_Date`
- `Actual_Completion_Date`
- `Treatment_Status`

## Cohort fields
- `Start_Quarter`
- `Start_Month`

## Drift analysis fields
- `Treatment_Drift_Days`
- `Treatment_Drift_Weeks`
- `On_Track_Status`

## Delay and correction fields
- `Delay_Reason`
- `Midcourse_Correction_Flag`

## Velocity and engagement fields
- `Treatment_Velocity`
- `Months_Since_Start`
- `Active_Patient_Flag`

## Final column list
- `Patient_ID`
- `Clinic_Location`
- `Start_Date`
- `Start_Quarter`
- `Start_Month`
- `Age_Group`
- `Projected_Treatment_Weeks`
- `Projected_Completion_Date`
- `Actual_Completion_Date`
- `Treatment_Status`
- `Treatment_Drift_Days`
- `Treatment_Drift_Weeks`
- `On_Track_Status`
- `Delay_Reason`
- `Midcourse_Correction_Flag`
- `Treatment_Velocity`
- `Months_Since_Start`
- `Active_Patient_Flag`

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

## Set seed for reproducibility

In [2]:
random.seed(42)
np.random.seed(42)

## Define dataset size and category values

In [3]:
n_records = 1200

clinic_locations = ["Downtown", "Northside", "West End", "Lakeside"]
age_groups = ["Teen", "Young Adult", "Adult", "Senior"]
delay_reasons = [
    "Missed Appointments",
    "Poor Aligner Compliance",
    "Refinement Needed",
    "Scheduling Delays",
    "Clinical Adjustment"
]

## Create helper functions

In [4]:
def random_start_date(start="2022-01-01", end="2023-12-31"):
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    delta_days = (end_dt - start_dt).days
    return start_dt + pd.to_timedelta(random.randint(0, delta_days), unit="D")

def assign_projected_weeks(age_group):
    if age_group == "Teen":
        return random.randint(36, 60)
    elif age_group == "Young Adult":
        return random.randint(30, 52)
    elif age_group == "Adult":
        return random.randint(28, 50)
    else:
        return random.randint(32, 56)

def assign_delay_reason(drift_weeks):
    if drift_weeks <= 0:
        return "None"
    return random.choices(
        delay_reasons,
        weights=[0.25, 0.30, 0.20, 0.15, 0.10],
        k=1
    )[0]

## Generate synthetic patient records

In [5]:
rows = []

today = pd.to_datetime("2024-12-31")

for i in range(1, n_records + 1):
    patient_id = f"PAT{i:04d}"
    clinic = random.choice(clinic_locations)
    age_group = random.choices(
        age_groups,
        weights=[0.35, 0.25, 0.30, 0.10],
        k=1
    )[0]

    start_date = random_start_date()
    projected_weeks = assign_projected_weeks(age_group)
    projected_completion = start_date + pd.to_timedelta(projected_weeks * 7, unit="D")

    # decide whether completed or still in progress
    completed_flag = random.choices([True, False], weights=[0.75, 0.25], k=1)[0]

    if completed_flag:
        # drift can be early, on time, or delayed
        drift_weeks = int(np.round(np.random.normal(loc=2, scale=4)))
        actual_completion = projected_completion + pd.to_timedelta(drift_weeks * 7, unit="D")
        status = "Completed"
    else:
        actual_completion = pd.NaT
        status = "In-Progress"
        drift_weeks = np.nan

    drift_days = drift_weeks * 7 if pd.notna(drift_weeks) else np.nan

    if pd.notna(drift_weeks) and drift_weeks <= 2:
        on_track_status = "On-Track"
    elif pd.notna(drift_weeks):
        on_track_status = "Delayed"
    else:
        on_track_status = "In-Progress"

    delay_reason = assign_delay_reason(drift_weeks) if pd.notna(drift_weeks) else "In-Progress"
    midcourse_flag = "Yes" if pd.notna(drift_weeks) and drift_weeks > 2 else "No"

    if status == "Completed":
        actual_weeks = max(1, ((actual_completion - start_date).days / 7))
        treatment_velocity = round(actual_weeks / projected_weeks, 2)
    else:
        treatment_velocity = np.nan

    start_quarter = f"Q{((start_date.month - 1) // 3) + 1} {start_date.year}"
    start_month = start_date.strftime("%b %Y")

    months_since_start = ((today.year - start_date.year) * 12) + (today.month - start_date.month)
    months_since_start = max(months_since_start, 0)

    active_flag = 1 if status == "In-Progress" else 0

    rows.append({
        "Patient_ID": patient_id,
        "Clinic_Location": clinic,
        "Start_Date": start_date.date(),
        "Start_Quarter": start_quarter,
        "Start_Month": start_month,
        "Age_Group": age_group,
        "Projected_Treatment_Weeks": projected_weeks,
        "Projected_Completion_Date": projected_completion.date(),
        "Actual_Completion_Date": actual_completion.date() if pd.notna(actual_completion) else "In-Progress",
        "Treatment_Status": status,
        "Treatment_Drift_Days": drift_days,
        "Treatment_Drift_Weeks": drift_weeks,
        "On_Track_Status": on_track_status,
        "Delay_Reason": delay_reason,
        "Midcourse_Correction_Flag": midcourse_flag,
        "Treatment_Velocity": treatment_velocity,
        "Months_Since_Start": months_since_start,
        "Active_Patient_Flag": active_flag
    })

## Convert to DataFrame

In [6]:
df = pd.DataFrame(rows)
df.head()

,Patient_ID,Clinic_Location,Start_Date,Start_Quarter,Start_Month,Age_Group,Projected_Treatment_Weeks,Projected_Completion_Date,Actual_Completion_Date,Treatment_Status,Treatment_Drift_Days,Treatment_Drift_Weeks,On_Track_Status,Delay_Reason,Midcourse_Correction_Flag,Treatment_Velocity,Months_Since_Start,Active_Patient_Flag
0,PAT0001,Downtown,2022-10-09,Q4 2022,Oct 2022,Teen,43,2023-08-06,2023-09-03,Completed,28.0,4.0,Delayed,Refinement Needed,Yes,1.09,26,0
1,PAT0002,Downtown,2022-02-02,Q1 2022,Feb 2022,Young Adult,30,2022-08-31,2022-09-07,Completed,7.0,1.0,On-Track,Missed Appointments,No,1.03,34,0
2,PAT0003,Downtown,2023-10-28,Q4 2023,Oct 2023,Young Adult,52,2024-10-26,2024-11-30,Completed,35.0,5.0,Delayed,Missed Appointments,Yes,1.10,14,0
3,PAT0004,West End,2022-01-07,Q1 2022,Jan 2022,Adult,33,2022-08-26,2022-10-21,Completed,56.0,8.0,Delayed,Poor Aligner Compliance,Yes,1.24,35,0
4,PAT0005,Northside,2022-12-11,Q4 2022,Dec 2022,Teen,39,2023-09-10,2023-09-17,Completed,7.0,1.0,On-Track,Missed Appointments,No,1.03,24,0


## Check the dataset shape and columns

## Quick validation checks

In [7]:
df["Clinic_Location"].value_counts()

,count
Clinic_Location,
West End,311
Downtown,307
Lakeside,296
Northside,286


In [8]:
df["Age_Group"].value_counts()

,count
Age_Group,
Teen,406
Adult,352
Young Adult,327
Senior,115


In [9]:
df["Treatment_Status"].value_counts()

,count
Treatment_Status,
Completed,909
In-Progress,291


In [10]:
df["On_Track_Status"].value_counts()

,count
On_Track_Status,
On-Track,500
Delayed,409
In-Progress,291


## Save the synthetic dataset

In [11]:
df.to_csv("orthodontic_treatment_velocity_synthetic.csv", index=False)
print("File saved: orthodontic_treatment_velocity_synthetic.csv")

File saved: orthodontic_treatment_velocity_synthetic.csv


In [12]:
from google.colab import files
files.download("orthodontic_treatment_velocity_synthetic.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>